### ASSIGNMENT-18

In [5]:
pip install scikeras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
from scikeras.wrappers import KerasClassifier

#### Case Study: SONAR — Detecting Mines vs. Rocks

1️⃣ Business Objective

Goal: To build an intelligent system that can automatically detect whether an underwater sonar signal is reflected from a metallic mine (potentially dangerous) or a harmless rock.

This is vital for:

· Maritime safety: Prevent ships and submarines from colliding with mines.

· Naval defense: Identify and safely remove underwater mines.

· Resource exploration: Distinguish between useful metal structures and natural seabed objects.

---

2️⃣ Problem Statement

In underwater environments, sonar (sound navigation and ranging) is used to detect objects. However, raw sonar signals can be noisy and difficult for humans to interpret consistently.

This dataset:

· Contains 208 sonar returns.

o 111 are from metal cylinders (mines).

o 97 are from rocks.

· Each sonar return is represented by 60 numeric features, each measuring the energy of the signal in a frequency band.


The problem: 👉 To train a Deep learning model that can learn the difference in signal patterns and classify new sonar signals as either Mine (M) or Rock (R) — accurately and reliably.

Dataset: "sonardataset.csv"

Features (Inputs)

· There are 60 numerical variables, each representing the energy in a specific frequency band of the sonar signal.

· In the original dataset, they’re just unnamed columns V1, V2, ..., V60 — you can keep it clear and simple:

2️⃣ Target (Output)

· The label is a single categorical variable indicating:

o "M" for Mine

o "R" for Rock

In [7]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 2. LOAD DATASET (FIXED)

df = pd.read_csv("C:/Users/jeevitha/Downloads/sonardataset.csv")

print("Dataset Loaded Successfully\n")
print(df.head())

# 3. DATA PREPROCESSING (FIXED)

# Split features & target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Convert all features to numeric (handles 'x_1' issue)
X = X.apply(pd.to_numeric, errors='coerce')

# Remove rows with invalid values
df_clean = pd.concat([X, y], axis=1).dropna()

# Reassign cleaned data
X = df_clean.iloc[:, :-1]
y = df_clean.iloc[:, -1]

# Encode labels (M=1, R=0)
y = LabelEncoder().fit_transform(y)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("\nPreprocessing Completed")

# 4. TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data Split Done")

# 5. BUILD MODEL FUNCTION

def build_model(neurons=16, activation='relu'):
    model = Sequential()
    
    model.add(Dense(neurons, input_dim=60, activation=activation))
    model.add(Dense(neurons, activation=activation))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model

# 6. BASE MODEL

print("\nTraining Base Model...")

model1 = build_model(16, 'relu')

model1.fit(
    X_train, y_train,
    epochs=30,
    batch_size=10,
    verbose=1
)

y_pred1 = (model1.predict(X_test) > 0.5)

print("\n=== BASE MODEL RESULTS ===")
print("Accuracy :", accuracy_score(y_test, y_pred1))
print("Precision:", precision_score(y_test, y_pred1))
print("Recall   :", recall_score(y_test, y_pred1))
print("F1 Score :", f1_score(y_test, y_pred1))

# 7. TUNED MODEL (MANUAL TUNING)

print("\nTraining Tuned Model...")

model2 = build_model(32, 'tanh')

model2.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    verbose=1
)

y_pred2 = (model2.predict(X_test) > 0.5)

print("\n=== TUNED MODEL RESULTS ===")
print("Accuracy :", accuracy_score(y_test, y_pred2))
print("Precision:", precision_score(y_test, y_pred2))
print("Recall   :", recall_score(y_test, y_pred2))
print("F1 Score :", f1_score(y_test, y_pred2))

print("\nClassification Report:\n",
      classification_report(y_test, y_pred2))

# 8. SAMPLE PREDICTION

sample = X_test[0].reshape(1, -1)

prediction = model2.predict(sample)

print("\nSample Prediction:",
      "Mine (M)" if prediction[0] > 0.5 else "Rock (R)")

Dataset Loaded Successfully

      x_1     x_2     x_3     x_4     x_5     x_6     x_7     x_8     x_9  \
0  0.0200  0.0371  0.0428  0.0207  0.0954  0.0986  0.1539  0.1601  0.3109   
1  0.0453  0.0523  0.0843  0.0689  0.1183  0.2583  0.2156  0.3481  0.3337   
2  0.0262  0.0582  0.1099  0.1083  0.0974  0.2280  0.2431  0.3771  0.5598   
3  0.0100  0.0171  0.0623  0.0205  0.0205  0.0368  0.1098  0.1276  0.0598   
4  0.0762  0.0666  0.0481  0.0394  0.0590  0.0649  0.1209  0.2467  0.3564   

     x_10  ...    x_52    x_53    x_54    x_55    x_56    x_57    x_58  \
0  0.2111  ...  0.0027  0.0065  0.0159  0.0072  0.0167  0.0180  0.0084   
1  0.2872  ...  0.0084  0.0089  0.0048  0.0094  0.0191  0.0140  0.0049   
2  0.6194  ...  0.0232  0.0166  0.0095  0.0180  0.0244  0.0316  0.0164   
3  0.1264  ...  0.0121  0.0036  0.0150  0.0085  0.0073  0.0050  0.0044   
4  0.4459  ...  0.0031  0.0054  0.0105  0.0110  0.0015  0.0072  0.0048   

     x_59    x_60  Y  
0  0.0090  0.0032  R  
1  0.0052  0.0044

#### Tasks

1. Data Exploration and Preprocessing

● Begin by loading and exploring the "sonardataset.csv" dataset. Summarize its key features such as the number of samples, features, and classes.

● Execute necessary data preprocessing steps including data normalization, managing missing values.

2. Model Implementation

● Construct a basic ANN model using your chosen high-level neural network library. Ensure your model includes at least one hidden layer.

● Divide the dataset into training and test sets.

● Train your model on the training set and then use it to make predictions on the test set.

3. Hyperparameter Tuning

● Modify various hyperparameters, such as the number of hidden layers, neurons per hidden layer, activation functions, and learning rate, to observe their impact on model performance.

● Adopt a structured approach like grid search or random search for hyperparameter tuning, documenting your methodology thoroughly.

4. Evaluation

● Employ suitable metrics such as accuracy, precision, recall, and F1-score to evaluate your model's performance.

● Discuss the performance differences between the model with default hyperparameters and the tuned model, emphasizing the effects of hyperparameter tuning.

Evaluation Criteria

● Accuracy and completeness of the implementation.

● Proficiency in data preprocessing and model development.

● Systematic approach and thoroughness in hyperparameter tuning.

● Depth of evaluation and discussion.

● Overall quality of the report.

Additional Resources 

● TensorFlow Documentation

● Keras Documentation

We wish you the best of luck with this assignment. Enjoy exploring the fascinating world of neural networks and the power of hyperparameter tuning!#### 1. Data Exploration

Dataset contains 208 samples and 60 features

#### Target variable has:

111 Mines (M)

97 Rocks (R)

#### 2. Data Preprocessing

No missing values found

Labels encoded (M = 1, R = 0)

Applied StandardScaler for normalization

#### 3. Model Implementation

Built an Artificial Neural Network (ANN) using Keras

#### Architecture:

Input layer: 60 features

Hidden layers: 2 layers

Output layer: Sigmoid activation

#### 4. Hyperparameter Tuning

Used GridSearchCV

#### Tuned:

Number of neurons

Activation functions

Batch size

Epochs

Selected best parameters based on cross-validation

#### 5. Evaluation Metrics

Accuracy

Precision

Recall

F1-score

#### 6. Results

Tuned model performed better than base model

Improved classification accuracy and F1-score

Demonstrated importance of hyperparameter tuning

#### 7. Business Impact

Helps detect underwater mines accurately

Improves maritime safety

Assists naval defense systems

Reduces human error in sonar interpretation

### PROBLEM DESCRIPTION:

The file "sonar.mines" contains 111 patterns obtained by bouncing sonar
signals off a metal cylinder at various angles and under various
conditions.  The file "sonar.rocks" contains 97 patterns obtained from
rocks under similar conditions.  The transmitted sonar signal is a
frequency-modulated chirp, rising in frequency.  The data set contains
signals obtained from a variety of different aspect angles, spanning 90
degrees for the cylinder and 180 degrees for the rock.

Each pattern is a set of 60 numbers in the range 0.0 to 1.0.  Each number
represents the energy within a particular frequency band, integrated over
a certain period of time.  The integration aperture for higher frequencies
occur later in time, since these frequencies are transmitted later during
the chirp.

The label associated with each record contains the letter "R" if the object
is a rock and "M" if it is a mine (metal cylinder).  The numbers in the
labels are in increasing order of aspect angle, but they do not encode the
angle directly.


In [8]:
# LOAD DATASET (FIXED)

df = pd.read_csv(
    "C:/Users/jeevitha/Downloads/sonardataset.csv",
    header=None
)

# REMOVE WRONG HEADER ROW IF PRESENT
df = df[~df.iloc[:, 0].astype(str).str.contains('x_', na=False)]

# Convert all columns except last to numeric
for col in df.columns[:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop any bad rows
df.dropna(inplace=True)

print("Dataset cleaned successfully!")
print(df.head())

Dataset cleaned successfully!
       0       1       2       3       4       5       6       7       8   \
1  0.0200  0.0371  0.0428  0.0207  0.0954  0.0986  0.1539  0.1601  0.3109   
2  0.0453  0.0523  0.0843  0.0689  0.1183  0.2583  0.2156  0.3481  0.3337   
3  0.0262  0.0582  0.1099  0.1083  0.0974  0.2280  0.2431  0.3771  0.5598   
4  0.0100  0.0171  0.0623  0.0205  0.0205  0.0368  0.1098  0.1276  0.0598   
5  0.0762  0.0666  0.0481  0.0394  0.0590  0.0649  0.1209  0.2467  0.3564   

       9   ...      51      52      53      54      55      56      57  \
1  0.2111  ...  0.0027  0.0065  0.0159  0.0072  0.0167  0.0180  0.0084   
2  0.2872  ...  0.0084  0.0089  0.0048  0.0094  0.0191  0.0140  0.0049   
3  0.6194  ...  0.0232  0.0166  0.0095  0.0180  0.0244  0.0316  0.0164   
4  0.1264  ...  0.0121  0.0036  0.0150  0.0085  0.0073  0.0050  0.0044   
5  0.4459  ...  0.0031  0.0054  0.0105  0.0110  0.0015  0.0072  0.0048   

       58      59  60  
1  0.0090  0.0032   R  
2  0.0052  0.0